# 06d - Task 3: Holdout evaluation + submission (LinearSVC)

Parallel variant of the 06/06b/06c holdout + submission process, applied to the tuned
LinearSVC from `05d_linearsvc_tuning.ipynb`.

**This notebook deliberately drops the threshold sweep that 06/06b/06c ran.** That is
the one substantive difference from its predecessors, and section 2 records why.

## 0. Setup

Imports, src helpers, paths.

In [1]:
# Adds project root to path so `import src...` works from notebooks/.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

import json
import joblib

from src import paths, data, evaluation

## 1. Load features, locked split, and tuned model

In [2]:
X, y, ids = data.load_train_features()
Xt, test_ids = data.load_test_features()
holdout_idx = np.load(paths.DATA_PROCESSED / 'holdout_idx.npy')
dev_idx = np.load(paths.DATA_PROCESSED / 'dev_idx.npy')

# Guard against a silent re-split.
assert len(dev_idx) == 16000 and len(holdout_idx) == 4000
assert not set(dev_idx) & set(holdout_idx), "dev and holdout overlap"

best = joblib.load(paths.MODELS / 'best_linearsvc_tuned.pkl')
with open(paths.MODELS / 'best_linearsvc_tuned.json') as f:
    tuned_meta = json.load(f)

print(f"Loaded {type(best).__name__}, fit on dev_idx only ({len(dev_idx)} rows)")
print(f"CV mean F1 from 05d: {tuned_meta['cv_mean_f1']:.4f} "
      f"(std {tuned_meta['cv_std_f1']:.4f}, "
      f"{tuned_meta['n_stage1_trials'] + tuned_meta['n_stage2_trials']} trials)")
print("Selected params:", tuned_meta["params"])
print(f"Selection rule: {tuned_meta['selection_rule']}")
print(f"(plain CV-argmax would have picked {tuned_meta['argmax_params']} "
      f"at {tuned_meta['argmax_cv_mean_f1']:.4f})")

Loaded LinearSVC, fit on dev_idx only (16000 rows)
CV mean F1 from 05d: 0.7275 (std 0.0040, 20 trials)
Selected params: {'C': 0.139824, 'loss': 'squared_hinge'}
Selection rule: smallest C among trials within 1 std of CV argmax
(plain CV-argmax would have picked {'C': 0.27964859516062457, 'loss': 'squared_hinge'} at 0.7276)


## 2. No threshold tuning - the question is closed

06/06b/06c each swept the decision threshold against out-of-fold CV probabilities and
emitted two candidate submissions. Three real Kaggle results have now settled it:

| model | tuned threshold | holdout gain | **Kaggle gain** |
|---|---|---|---|
| lightgbm | 0.525 (up) | +0.0031 | **+0.00764** |
| logreg_ridge | 0.475 (down) | +0.0065 | **-0.01576** |
| logreg_elasticnet | 0.460 (down) | -0.0016 | **-0.01999** |

Both linear models' tuned thresholds moved *below* 0.5, pushing predictions further
toward the machine class and away from train's 62.5% balance, and both lost on the real
leaderboard - ElasticNet by 0.02 (0.69576 down to 0.67577), the largest miss yet.
Holdout predicted the wrong sign for Ridge and understated the damage for ElasticNet.

The reason is that the threshold is tuned on data drawn from the
*train* distribution, and it is precisely a lever on predicted class balance, which is
the axis the documented train-to-test shift runs along. A correction that is optimal for
the train distribution is close to guaranteed to be wrong for the shifted one.

LinearSVC is a linear model, so it falls squarely in the family where this backfired
twice out of two. **Predictions are emitted at the default cutoff only** - for LinearSVC
that means `decision_function >= 0`, the max-margin boundary itself.

This also sidesteps the fact that LinearSVC has no `predict_proba`; the calibrated
probability version is built in `07_linear_ensemble.ipynb`, where soft voting needs it.

## 3. Holdout Macro F1

In [3]:
holdout_pred = best.predict(X[holdout_idx])
holdout_f1 = evaluation.macro_f1(y[holdout_idx], holdout_pred)

cv_f1 = tuned_meta["cv_mean_f1"]
gap = cv_f1 - holdout_f1

print(f"CV mean F1 (05d): {cv_f1:.4f}")
print(f"Holdout Macro F1 (decision_function >= 0): {holdout_f1:.4f}")
print(f"Gap (CV - holdout): {gap:+.4f}")

# Sanity check the margin distribution - a healthy linear SVM should have scores
# spread on both sides of 0, not piled up on one side.
margins = best.decision_function(X[holdout_idx])
print(f"\nHoldout decision_function: min {margins.min():.3f}, "
      f"median {np.median(margins):.3f}, max {margins.max():.3f}")
print(f"Predicted machine-class share: {holdout_pred.mean():.4f} "
      f"(true holdout share {y[holdout_idx].mean():.4f})")

CV mean F1 (05d): 0.7275
Holdout Macro F1 (decision_function >= 0): 0.7414
Gap (CV - holdout): -0.0139



Holdout decision_function: min -1.858, median 0.059, max 1.716
Predicted machine-class share: 0.5567 (true holdout share 0.6252)


In [4]:
# Ledger-safe write: replace this model's row if it already exists rather than
# overwriting other models' rows - holdout_metrics.csv accumulates one row per model.
# The threshold_* columns stay NaN here by design: no threshold was tuned (section 2).
metrics_path = paths.DATA_PROCESSED / "holdout_metrics.csv"
new_row = pd.DataFrame([{
    "model": "linearsvc",
    "cv_mean_f1": cv_f1,
    "holdout_f1": holdout_f1,
    "gap": gap,
}])
if metrics_path.exists():
    existing = pd.read_csv(metrics_path)
    existing = existing[existing["model"] != "linearsvc"]
    holdout_metrics = pd.concat([existing, new_row], ignore_index=True)
else:
    holdout_metrics = new_row
holdout_metrics.to_csv(metrics_path, index=False)
print(holdout_metrics.to_string(index=False))

            model  cv_mean_f1  holdout_f1       gap  threshold  cv_f1_at_threshold  holdout_f1_at_threshold  threshold_gain_holdout
         lightgbm    0.744273    0.743691  0.000582      0.525            0.746203                 0.746767                0.003076
     logreg_ridge    0.727866    0.739153 -0.011287      0.475            0.730510                 0.745662                0.006509
logreg_elasticnet    0.728985    0.739529 -0.010544      0.460            0.729834                 0.737922               -0.001607
        linearsvc    0.727490    0.741435 -0.013945        NaN                 NaN                      NaN                     NaN


## 4. Refit on full train + write submission

Refits once on all labeled data (dev + holdout); the holdout split's only job was the
honest check above. One submission file, default cutoff.

In [5]:
best.fit(X, y)
joblib.dump(best, paths.MODELS / "final_linearsvc_full_refit.pkl")

preds = best.predict(Xt).astype(int)

# Assert alignment BEFORE writing - a misaligned id column silently destroys the
# submission and the leaderboard gives no hint that it happened.
sample = pd.read_csv(paths.DATA_RAW / "sample_submission.csv", dtype={"id": str})
assert len(test_ids) == 6999, len(test_ids)
assert list(test_ids) == list(sample["id"]), "test ids do not match sample_submission order"

data.write_submission(test_ids, preds, "linearsvc_task3_default_threshold.csv")

bal = pd.Series(preds).value_counts(normalize=True).round(4).to_dict()
train_bal = pd.Series(y).value_counts(normalize=True).round(4).to_dict()
print(f"Wrote linearsvc_task3_default_threshold.csv - holdout F1 {holdout_f1:.4f}")
print(f"  predicted balance {bal}")
print(f"  train class balance {train_bal}")

Wrote linearsvc_task3_default_threshold.csv - holdout F1 0.7414
  predicted balance {1: 0.663, 0: 0.337}
  train class balance {1: 0.6252, 0: 0.3748}


## Discussion / carry-forward -> `07_linear_ensemble.ipynb`

**Holdout Macro F1 0.7414 at the default cutoff, the best holdout figure of any linear
model tried** (ElasticNet 0.7395, Ridge 0.7392) despite LinearSVC having the *lowest* CV
of the three (0.7275). Its CV-to-holdout gap is -0.0139, the largest negative gap in the
project, meaning it generalized further beyond its own CV estimate than anything else.
That is consistent with it being the most heavily regularized model in the set.

**The class-balance observation that motivated `08_calibration.ipynb` starts here.**
LinearSVC predicts only 55.67% machine on the holdout against a true share of 62.52%, so
it under-predicts the majority class, while the full-refit model predicts 66.30% machine
on the Kaggle test set. The same model leans one way on train-distribution data and the
other way on test, which is direct evidence of the shift rather than an inference about
it. Once the six scored submissions were compared, predicted machine share turned out to
rank the leaderboard perfectly, so that observation became the subject of notebook 08.

**No threshold sweep was run,** for the reasons in section 2: both linear models' CV-tuned
thresholds moved below 0.5 and both lost on Kaggle, ElasticNet's by 0.02. Notebook 08
revisits the threshold as a *calibration* lever aimed at class balance, which is the
opposite direction and is driven by leaderboard evidence rather than by holdout data.

**Carry forward:** `submissions/linearsvc_task3_default_threshold.csv` (predicted share
0.6630) is the first of the four uploads planned in 08, where it serves as the control -
nearly the same share as the best-scoring ElasticNet submission but a different model and
a different objective.